# Massive CV Ensemble
* 5 folds x 40 seeds x 5 Configs GBSA Model
* Use GBSA and LGBM

In [ ]:
import numpy as np
import pandas as pd
import warnings
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, RepeatedKFold
from sksurv.ensemble import GradientBoostingSurvivalAnalysis
from sksurv.util import Surv
from sklearn.metrics import (
    concordance_index_censored,
    brier_score_loss,
    intergrated_brier_score
)
import sys
import platform
import sklearn
import sksurv
import kagglehub
import os
from Config import Config
from dataclasses import asdict
from Config import MetricOuput

warnings.filterwarnings("ignore")

# Enviorment report

In [49]:

print("Python: ", sys.version.split()[0])
print("OS: ", platform.platform())
print("Scikit-learn: ", sklearn.__version__)
print("Scikit-survival: ", sksurv.__version__)

Python:  3.10.0
OS:  Windows-10-10.0.26200-SP0
Scikit-learn:  1.7.2
Scikit-survival:  0.25.0


# Paths

In [50]:
COMP_DIR = kagglehub.competition_download('WiDSWorldWide_GlobalDathon26')
metadata_path = os.path.join(COMP_DIR, 'metaData.csv')
train_path = os.path.join(COMP_DIR, 'train.csv')
test_path = os.path.join(COMP_DIR, 'test.csv')

print("Metadata path: ", metadata_path)
print("Train path: ", train_path)
print("Test path: ", test_path)

Metadata path:  C:\Users\kdch6\.cache\kagglehub\competitions\WiDSWorldWide_GlobalDathon26\metaData.csv
Train path:  C:\Users\kdch6\.cache\kagglehub\competitions\WiDSWorldWide_GlobalDathon26\train.csv
Test path:  C:\Users\kdch6\.cache\kagglehub\competitions\WiDSWorldWide_GlobalDathon26\test.csv


# Utility

In [ ]:
HORIZONS_PRED = np.array([12, 24, 48, 72])  # 예측할 시간 간격 (시간 단위)

from sklearn.base import BaseEstimator

def set_seed(seed=42):
    np.random.seed(seed)

import numpy as np
from sksurv.metrics import concordance_index_censored, integrated_brier_score

def evaluate_hybrid(model, X_train, y_train, X_valid, y_valid, times) -> MetricOuput:
    # 1) Risk scores for C-index
    risk_scores = model.predict(X_valid)

    c_index = concordance_index_censored(
        y_valid["event"],
        y_valid["time"],
        risk_scores
    )[0]

    # 2) Survival probabilities for IBS
    surv_fns = model.predict_survival_function(X_valid)
    surv_preds = np.row_stack([fn(times) for fn in surv_fns])  # S(t)

    ibs = integrated_brier_score(
        y_train,
        y_valid,
        surv_preds,
        times
    )

    hybrid_score = 0.3 * c_index + 0.7 * (1.0 - ibs)
    result = MetricOuput(
        c_index=c_index,
        ibs=ibs,
        hybrid_score=hybrid_score,
        risk_scores=risk_scores,
        surv_preds=surv_preds,
        hit_probs=1.0 - surv_preds
    )
    return result

def get_surv_predictions(model, X)-> np.ndarray:
    surv_fns = model.predict_survival_function(X)
    preds = np.empty((len(surv_fns), len(HORIZONS_PRED)), dtype=float)
    for i, fn in enumerate(surv_fns):
        t_min, t_max = fn.domain
        preds[i, :] = fn(np.clip(HORIZONS_PRED, t_min, t_max))
    return 1.0 - preds

def KFold_val(model: BaseEstimator, data:pd.DataFrame, seed, n_splits=5, n_repeats=2, ):
    KFold = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)
    scores = MetricOuput(c_index=0.0, ibs=0.0, hybrid_score=0.0, risk_scores=None, surv_preds=None, hit_probs=None)
    for fold, (train_idx, val_idx) in enumerate(KFold.split(data)):
        print(f"Fold {fold + 1}")
        X_train = data.iloc[train_idx].drop(columns=['id', 'time_to_hit_hours'])
        y_train = Surv.from_arrays(data.iloc[train_idx]['time_to_hit_hours'] > 0, data.iloc[train_idx]['time_to_hit_hours'])
        X_val = data.iloc[val_idx].drop(columns=['id', 'time_to_hit_hours'])
        y_val = Surv.from_arrays(data.iloc[val_idx]['time_to_hit_hours'] > 0, data.iloc[val_idx]['time_to_hit_hours'])
        
        model.fit(X_train, y_train)

        # 평가 지표 계산 및 출력 (예: C-index)
        # c_index = concordance_index_censored(y_val['event'], y_val['time'], val_pred[:, 0])  
        result = evaluate_hybrid(model, X_train, y_train, X_val, y_val, HORIZONS_PRED)
        scores.hybrid_score += result.hybrid_score
        scores.c_index += result.c_index
        scores.weighted_brier += result.weighted_brier

        print(f"Fold{1+fold} Hybrid Score: ", result.hybrid_score)
        print(f"Fold{1+fold} C-index: ", result.c_index)
        print(f"Fold{1+fold} Brier Score: ", result.weighted_brier)

    avg_score = scores / (n_splits * n_repeats)
    return avg_score



## Feature Engineering

## Basic Parameters
### 1. `dist`
```python
dist = result["dist_min_ci_0_5h"].clip(lower=1)
```

* 초기 0~5h동안 대피구역까지의 최소거리 $m$단위
* `clip(lower=1)`은 거리가 0까지 내려가지 않도록 최소 1로 잘라서, 후에 있을 $\log$, $\frac{1}{dist}$계산이 가능하도록 함

### 2. `speed`
```python
speed = result["closing_speed_m_per_h"]
```

* 산불이 대피구역 방향으로 얼마나 빠르게 접근하는지를 나타내는 속도

### 3. `perimeters`
```python
perimeters = result["num_perimeters_0_5h"]
```

* 초기 0~5시간 동안 관측된 perimeter(화재 외곽선) 개수를 가져옴. -> 정확히 무엇을 의미하는지 추가 조사 필요
* 보통 perimeter가 많으면 화재 활동성이 더 크다고 볼 수 있음.

### 4. `area_first`

```python
area_first = result["area_first_ha"]
```

* 초기 화재 면적(ha 단위)

### 5. `fire_radius`
```python
fire_radius = np.sqrt(area_first * 10000 / np.pi)
```
* 초기 화재 발생면적이 원이라고 가정했을 때의 초기 화재 발생반경
* $1ha = 10,000m^2$이므로 `area_first` $\times 10,000$으로 단위를 $ha$에서 $m^2$로 변환 후, 원의 넓이 공식 $A= \pi r^2$에서 $r = \sqrt(\frac{A}{\pi})$ 사용
---
## Distance transformations
### 1. `"log_distance"`
```python
result["log_distance"] = np.log1p(dist)
```

* $\log(1+dist)$으로 `dist`를 변환
* $\log$인지라 작은 거리는 차이를 크게 큰 거리는 차이를 작게 변환 
* $\log(1+dist)$라서 0 근처도 안전함.

### 2. `"inv_distance"`
```python
result["inv_distance"] = 1 / (dist / 1000 + eps)
```
**위험도를 나타내는 피처**
* 거리의 역수
* 가까울수록 값은 커짐
* 1000으로 나누어 $m$를 $km$로 eps은 0으로 나누는 것을 방지

### 3. `"sqrt_distance"`
```python
result["sqrt_distance"] = np.sqrt(dist)
```
* `dist`의 제곱근 변환
* 로그변환보다는 약하게, 원본보다는 강하게 거리의 차이를 좁힘

### 4. `"dist_km_sq"`
```python
result["dist_km_sq"] = (dist / 1000) ** 2
```
* 거리(km)의 제곱.
* 거리와 위험도의 비선형 관계를 모델이 더 쉽게 잡도록 도와줌.

### 5. `"dist_rank"`
```python
result["dist_rank"] = dist.rank(pct=True)
```

* 각 샘플의 거리가 전체 데이터에서 어느 정도 순위인지 백분위로 표현. -> 실제 테스트에서는 전체 데이터를 train을 포함한 걸로 해야 할지 test만 해야 할지
* 절대 거리 대신 상대적으로 “얼마나 가까운 편인가”를 넣는 것.

---
## Area-to-distance
### 1. `"radius_to_dist"`
```python
result["radius_to_dist"] = fire_radius / dist
```
* 화재 반경이 대피구역까지 거리 대비 얼마나 큰지 나타냄.
* 불이 크고 대피구역이 가까울수록 값이 커짐.

### 2. `"area_to_dist_ratio"`
```python
result["area_to_dist_ratio"] = area_first / (dist / 1000 + 0.1)
```

* 화재 면적을 거리(km)로 나눈 값.
* 큰 화재가 가까이 있을수록 위험하다는 아이디어.

### 3. `"log_area_dist_ratio"`
```python
result["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)
```
* 로그 공간에서 면적과 거리의 차이를 표현.

* 큰 화재 + 짧은 거리 조합을 더 안정적으로 나타냄. (산불이 진행해야 할 거리 표현)

* 사실상 $\log(\frac{1+A}{1+d})$와 동일 따라서 아래와 같이 바꿀 수 있음
    ```python
    np.log((1+area_first)/(1+dist))
    ```
---
## Movement / kinematics
### 1. `"has_movement"`
```python
result["has_movement"] = (perimeters > 1).astype(float)
```
* `perimeters`가 1개보다 많으면 “움직임/확장 있음”으로 보고 1, 아니면 0으로 둠.
* 이진 피처로 단순화해서 모델이 쓰기 쉽게 만든 것.

### 2. `closing_pos`
```python
closing_pos = speed.clip(lower=0)
```

* 접근 속도가 음수면(멀어지는 방향이면) 0으로 잘라버림.
* 즉 실제 “다가오는 속도”만 남김. -> 그럼 아예 조건식으로 얘가 0이면 prob를 다 0으로 출력해도 되지 않나?

### 3. `"eta_hours"`
```python
result["eta_hours"] = np.where(closing_pos > min_speed, dist / closing_pos, max).clip(max=max_hours)
```
* `dist / speed` 형태.
* 속도가 너무 작거나 0 (0.01보다 작으면)이면 사실상 도달 안 하는 것으로 보고 `max_hours`(9999)를 넣음.
* 마지막 `clip(max=max_hours)`는 너무 큰 값도 제한.

## Movement / Kinematics Features

### `log_eta`

```python
result["log_eta"] = np.log1p(result["eta_hours"].clip(0, 9999))
```

ETA를 로그 변환해서 스케일을 줄인다.

- ETA 값이 매우 커질 수 있기 때문에 로그 변환으로 스케일을 압축
- 매우 큰 ETA 값들의 영향력을 줄여 모델 학습 안정성을 높임

---

### `radial_growth`

```python
radial_growth = result["radial_growth_rate_m_per_h"].clip(lower=0)
```

화재가 **반경 방향으로 퍼지는 속도**를 가져온다.

- 음수 값은 화재가 줄어드는 경우이므로 0으로 잘라낸다
- 즉 화재의 **확산만 반영**

---

### `effective_closing_speed`

```python
effective_closing = closing_pos + radial_growth
result["effective_closing_speed"] = effective_closing
```

단순 접근 속도와 화재 확산 속도를 합친 값이다.

- `closing_pos`: 화재의 접근 속도
- `radial_growth`: 화재 확산 속도

즉 **화재 이동 + 화재 확산**을 동시에 고려한 실제 접근 속도.

---

### `eta_effective`

```python
result["eta_effective"] = np.where(
    effective_closing > 0.01,
    dist / effective_closing,
    9999
).clip(max=9999)
```

확산까지 고려한 ETA(도달 예상 시간).

- 단순 이동 기반 ETA보다 **물리적으로 더 의미 있는 값**
- 확산까지 고려하여 **실제 화재 도달 가능성**을 더 잘 반영

---

### `threat_score`

```python
result["threat_score"] = result["alignment_abs"] * speed / np.log1p(dist)
```

화재 위협 점수.

다음 세 요소를 동시에 반영한다.

- `alignment_abs`: 화재 이동 방향이 대피구역 방향과 얼마나 정렬되어 있는지
- `speed`: 화재 접근 속도
- `dist`: 대피구역까지 거리

즉 다음 조건일수록 값이 커진다.

- 대피구역 방향으로 이동
- 접근 속도가 빠름
- 거리가 가까움

→ **대피구역 쪽으로 빠르게, 가까이 접근하는가**를 표현하는 feature.

---

### `fire_urgency`

```python
result["fire_urgency"] = perimeters * speed
```

화재의 **활동성 + 접근 속도**를 동시에 반영하는 feature.

- perimeter 수가 많고
- 접근 속도가 빠를수록

값이 커진다.

즉 화재 상황이 **얼마나 급박한지**를 나타낸다.

---

### `growth_intensity`

```python
result["growth_intensity"] = result["area_growth_rate_ha_per_h"] * perimeters
```

화재 성장 강도.

- 면적 증가 속도
- perimeter 수

를 결합한 feature로 **화재가 얼마나 활발하게 성장하고 있는지**를 나타낸다.

---

## Distance Zone Features

### 1. `"zone_critical"`

```python
result["zone_critical"] = (dist < 5000).astype(float)
```

대피구역까지 거리가 **5km 미만**이면 1.

→ **가장 위험한 구간**

---

### 2. `"zone_warning"`

```python
result["zone_warning"] = ((dist >= 5000) & (dist < 10000)).astype(float)
```

거리 **5km ~ 10km**

→ **경고 구간**

---

### 3. `"zone_safe"`

```python
result["zone_safe"] = (dist >= 10000).astype(float)
```

거리 **10km 이상**

→ **상대적으로 안전한 구간**

---

이 세 feature는 **거리의 비선형 임계값 효과**를 모델이 쉽게 학습하도록 돕는다.

예를 들어:

- 3km vs 8km
- 8km vs 15km

이 차이는 단순 거리 차이 이상의 의미를 가질 수 있기 때문에  
**zone 기반 feature**를 추가한다.

---

## Temporal Features

### 1. `"is_summer"`

```python
result["is_summer"] = result["event_start_month"].isin([6, 7, 8]).astype(float)
```

화재 시작 월이 **6, 7, 8월이면 여름**으로 판단.

여름에는

- 기온 상승
- 건조 환경
- 산불 확산 위험 증가

등의 요인이 있을 수 있기 때문에 이를 반영한다.

---

### 2. `"is_afternoon"`

```python
result["is_afternoon"] = (
    (result["event_start_hour"] >= 12) &
    (result["event_start_hour"] < 20)
).astype(float)
```

화재 시작 시간이 **12시 ~ 20시 사이**이면 오후로 판단.

시간대에 따라

- 온도
- 풍속
- 대기 상태

등이 달라질 수 있으며  
이는 화재 확산 패턴에 영향을 줄 수 있다.
## 의문점
### 1. perimeters가 무엇을 의미하는지

### 2. `dist_rank`를 계산할 때 후에 실제 predict에서는 test만 넣어야 하는지 아니면 train도 넣어야 하는지

### 3. `"log_area_dist_ratio"`는 `fire_radius`가 아니라 면적이 들어가는지

In [52]:
def create_features(df: pd.DataFrame, eps=0.1, max_hours=9999, min_speed=0.01)-> pd.DataFrame:
    """
    Args:
        df (pd.DataFrame): Input DataFrame with raw features.
        eps (float): Small constant to avoid division by zero in transformations.
        max_hours (int): Maximum hours to cap time-based features for stability.
    Returns:
        pd.DataFrame: DataFrame with engineered features.
        
    ---
   

    ### 
    """
    one_km = 1000
    one_ha = 10000
    result = df.copy()
    dist = result['dist_min_ci_0_5h'].clip(lower=1)
    speed = result['closing_speed_m_per_h']
    perimeters = result['num_perimeters_0_5h']
    area_first = result['area_first_ha']
    result['log_distance'] = np.log1p(dist)
    result['inv_distance'] = 1 / (dist / one_km + eps)
    result['inv_distance_sq'] = result['inv_distance'] ** 2
    result['sqrt_distance'] = np.sqrt(dist)
    result['dist_km'] = dist / one_km
    result['dist_km_sq'] = (dist / one_km) ** 2
    result['dist_rank'] = dist.rank(pct=True)
    fire_radius = np.sqrt(area_first * one_ha / np.pi)
    result['radius_to_dist'] = fire_radius / dist
    result['area_to_dist_ratio'] = area_first / (dist / one_km + eps)
    result['log_area_dist_ratio'] = np.log1p(area_first) - np.log1p(dist)
    result['has_movement'] = (perimeters > 1).astype(float)
    closing_pos = speed.clip(lower=0)
    result['eta_hours'] = np.where(closing_pos > min_speed, dist / closing_pos, max_hours).clip(max=max_hours)
    result['log_eta'] = np.log1p(result['eta_hours'].clip(0, max_hours))
    radial_growth = result['radial_growth_rate_m_per_h'].clip(lower=0)
    effective_closing = closing_pos + radial_growth
    result['effective_closing_speed'] = effective_closing
    result['eta_effective'] = np.where(effective_closing > min_speed, dist / effective_closing, max_hours).clip(max=max_hours)
    result['threat_score'] = result['alignment_abs'] * speed / np.log1p(dist)
    result['fire_urgency'] = perimeters * speed
    result['growth_intensity'] = result['area_growth_rate_ha_per_h'] * perimeters
    result['zone_critical'] = (dist < 5000).astype(float)
    result['zone_warning'] = ((dist >= 5000) & (dist < 10000)).astype(float)
    result['zone_safe'] = (dist >= 10000).astype(float)
    result['is_summer'] = result['event_start_month'].isin([6, 7, 8]).astype(float)
    result['is_afternoon'] = ((result['event_start_hour'] >= 12) & (result['event_start_hour'] < 20)).astype(float)
    drop_cols = ['relative_growth_0_5h', 'projected_advance_m', 'centroid_displacement_m',
                 'centroid_speed_m_per_h', 'closing_speed_abs_m_per_h', 'area_growth_abs_0_5h']
    result = result.drop(columns=[c for c in drop_cols if c in result.columns])
    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)
    return result



# Data

In [53]:
metadata_df = pd.read_csv(metadata_path)
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

In [54]:
train_df.head()

,event_id,num_perimeters_0_5h,dt_first_last_0_5h,low_temporal_resolution_0_5h,area_first_ha,area_growth_abs_0_5h,area_growth_rel_0_5h,area_growth_rate_ha_per_h,log1p_area_first,log1p_growth,...,dist_fit_r2_0_5h,alignment_cos,alignment_abs,cross_track_component,along_track_speed,event_start_hour,event_start_dayofweek,event_start_month,time_to_hit_hours,event
0,10892457,3,4.265188,0,79.696304,2.875935,0.036086,0.674281,4.390693,1.354787,...,0.886373,-0.054649,0.054649,-1.937219,-0.106026,19,4,5,18.892512,0
1,11757157,2,1.169918,0,8.946749,0.000000,0.000000,0.000000,2.297246,0.000000,...,0.000000,-0.568898,0.568898,-0.000000,-0.000000,4,4,6,22.048108,1
2,11945086,4,4.777526,0,106.482638,0.000000,0.000000,0.000000,4.677329,0.000000,...,0.000000,0.882385,0.882385,0.000000,0.000000,22,4,8,0.888895,1
3,12044083,1,0.000000,1,67.631125,0.000000,0.000000,0.000000,4.228746,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,20,5,8,60.953021,0
4,12052347,2,4.975273,0,35.632874,0.000000,0.000000,0.000000,3.600946,0.000000,...,0.000000,0.934634,0.934634,-0.000000,0.000000,21,5,7,44.990274,0


In [55]:
train_df['time_to_hit_hours'].max()

66.99447413277778

# Baseline (Gradient Boosting Survival Analysic)

## 1. Preprocessing

In [ ]:
from pprint import pprint

cfg = Config()
gbsa_params = asdict(cfg.gbsa_config)
preprocessing_params = asdict(cfg.preprocessing_config)
seed = cfg.seed
set_seed(seed)

train_processed = create_features(train_df, **preprocessing_params)
test_processed = create_features(test_df, **preprocessing_params)

## 2. Create Model (GBSA)

In [ ]:
gbsa = GradientBoostingSurvivalAnalysis(**gbsa_params)
print("Model parameters: ")
pprint(gbsa.get_params())

Model parameters: 
{'ccp_alpha': 0.0,
 'criterion': 'friedman_mse',
 'dropout_rate': 0.0,
 'learning_rate': 0.1,
 'loss': 'coxph',
 'max_depth': 3,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'n_estimators': 100,
 'n_iter_no_change': None,
 'random_state': 42,
 'subsample': 1.0,
 'tol': 0.0001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}


## 3. RepeatedCV

array([(False, 18.89251185), ( True, 22.04810801), ( True,  0.88889548),
       (False, 60.95302147), (False, 44.99027447)],
      dtype=[('event', '?'), ('time_to_hit_hours', '<f8')])